In [49]:
from typing import Iterable
import os
import shutil
import librosa
import soundfile
import torch
import numpy as np
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline, Pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
model_id = "openai/whisper-small"

all_augmentations = ['noise', 'squish', "stretch"]


def create_dataset(source_folder: str, output_folder: str, augmentations: list[str]):
    sources = [os.path.join(source_folder, source) for source in os.listdir(source_folder)]
    print("Found files:")
    print(sources)

    if os.path.exists(output_folder):
        shutil.rmtree(output_folder)
    os.mkdir(output_folder)

    annotated_data = annotate_data(sources)  # (filename + annotation)[]
    base_dataset = generate_base_dataset(annotated_data)
    augmented_dataset = augment_dataset(base_dataset, augmentations)

    save_dataset(augmented_dataset, output_folder)


def create_pipeline() -> Pipeline:
    model = AutoModelForSpeechSeq2Seq.from_pretrained(
        model_id, dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True
    )
    model.to(device)

    processor = AutoProcessor.from_pretrained(model_id)

    generation_config = {
        "num_beams": 1,
        "condition_on_prev_tokens": False,
        "compression_ratio_threshold": 1.35,  # zlib compression ratio threshold (in token space)
        "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
        "logprob_threshold": -1.0,
        "no_speech_threshold": 0.6,
        "return_timestamps": True,
    }

    return pipeline(
        "automatic-speech-recognition",
        model=model,
        tokenizer=processor.tokenizer,
        feature_extractor=processor.feature_extractor,
        dtype=dtype,
        device=device,
        config=generation_config,
        return_timestamps=True
    )


## parses text from file and returns:
# (
#     filename,
#     {
#         test: string,
#         chunks: {
#             timestamp: (number, number),
#             text: string
#         }[]
#     }
# )[]
# TODO: remove directory reading from method. At most it should only read from passed paths
def annotate_data(sources: list[str]) -> Iterable[tuple[str, dict]]:
    pipe = create_pipeline()

    for source_path in sources:
        print(f"Annotating {source_path}")
        source_annotation = pipe(source_path)
        yield source_path, source_annotation


type DatasetItem = tuple[str, str, np.ndarray, int | float, str]


def generate_base_dataset(annotated_data) -> Iterable[DatasetItem]:
    for (source_path, annotation) in annotated_data:
        print(f"Processing {source_path}")
        file, sampling_rate = librosa.load(source_path, sr=16000)
        text, chunks = annotation["text"], annotation["chunks"]

        for i, chunk in enumerate(chunks):
            chunk_text, (start, end) = chunk["text"], chunk["timestamp"]
            start_index = int(start * sampling_rate)
            end_index = int(end * sampling_rate)

            chunk_audio = file[start_index:end_index]
            filename, file_extension = os.path.splitext(source_path)
            base_name = os.path.basename(filename)

            chunk_filename = f"{base_name}_{i}"

            yield chunk_filename, file_extension, chunk_audio, sampling_rate, chunk_text


# accepts (filename_base, librosa audio file object, sample rate,  parsed text)
def augment_dataset(base_dataset: Iterable[DatasetItem], augmentations: list[str]) -> Iterable[DatasetItem]:
    augmented_dataset = base_dataset
    augmented_dataset = augment_stretch_squish(augmented_dataset, augmentations)
    augmented_dataset = augment_noise(augmented_dataset, augmentations)

    yield from augmented_dataset


def augment_stretch_squish(base_dataset: Iterable[DatasetItem], augmentations: list[str]):
    if "stretch" not in augmentations and "squish" not in augmentations:
        yield from base_dataset

    for item in base_dataset:
        yield item
        if "stretch" in augmentations:
            yield stretch_augment(item)
        if "squish" in augmentations:
            yield squish_augment(item)


def stretch_augment(item: DatasetItem) -> DatasetItem:
    chunk_filename, file_extension, chunk_audio, sampling_rate, chunk_text = item
    augmented_audio = librosa.effects.time_stretch(chunk_audio, rate=0.97)

    return chunk_filename + "_stretch", file_extension, augmented_audio, sampling_rate, chunk_text


def squish_augment(item: DatasetItem) -> DatasetItem:
    chunk_filename, file_extension, chunk_audio, sampling_rate, chunk_text = item
    augmented_audio = librosa.effects.time_stretch(chunk_audio, rate=1.03)

    return chunk_filename + "_squish", file_extension, augmented_audio, sampling_rate, chunk_text


def augment_noise(base_dataset: Iterable[DatasetItem], augmentations: list[str]):
    if "noise" not in augmentations:
        yield from base_dataset

    for item in base_dataset:
        yield item
        yield noise_augment(item)


def noise_augment(item: DatasetItem) -> DatasetItem:
    chunk_filename, file_extension, chunk_audio, sampling_rate, chunk_text = item
    STD_n = 0.001
    noise = np.random.normal(0, STD_n, chunk_audio.shape[0])

    augmented_audio = chunk_audio + noise

    return chunk_filename + "_noise", file_extension, augmented_audio, sampling_rate, chunk_text


def save_dataset(dataset: Iterable[DatasetItem], output_folder: str):
    index = dict()
    for item in dataset:
        chunk_filename, file_extension, chunk_audio, sampling_rate, chunk_text = item
        filename = os.path.normpath(os.path.join(output_folder, chunk_filename + file_extension))

        index[filename] = chunk_text
        soundfile.write(filename, chunk_audio, samplerate=int(sampling_rate))

    with open(os.path.join(output_folder, "index.json"), "w", encoding="utf-8") as file:
        file.write(str(index).replace("'", '"'))


create_dataset("./sonya_source", "./sonya_dataset", ["noise"])

Found files:
['./sonya_source\\interview-1.wav', './sonya_source\\Sonya_1.wav', './sonya_source\\Sonya_2.wav', './sonya_source\\sonya_rim.wav']


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Annotating ./sonya_source\interview-1.wav
Processing ./sonya_source\interview-1.wav
Annotating ./sonya_source\Sonya_1.wav
Processing ./sonya_source\Sonya_1.wav
Annotating ./sonya_source\Sonya_2.wav
Processing ./sonya_source\Sonya_2.wav
Annotating ./sonya_source\sonya_rim.wav
Processing ./sonya_source\sonya_rim.wav
